# Causal tracing: production run and results

This notebook does not reimplement causal tracing. It invokes the same Hydra command used by the CLI, then reads and visualizes the production artifacts. See causal_tracing.md for the numerical contract.

## 1. Imports and repository root

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.main import run_hydra

## 2. Production settings

Edit MODEL or OVERRIDES. RUN_TRACE stays false so opening the notebook never downloads a model or starts a GPU job.

In [ ]:
MODEL = "gpt2-large"
RUN_TRACE = False
OVERRIDES = [
    "command.causal_trace.num_valid_facts=100",
    "command.causal_trace.overwrite_model_config_layer=false",
]
OUTPUT_ROOT = ROOT / "analysis_out" / "causal_trace"
RUN_DIR = None  # Set to a specific completed run directory when needed.

display(pd.Series({
    "model": MODEL,
    "accepted facts": 100,
    "noise draws per fact": 10,
    "window width": 10,
    "discovery fraction": 0.5,
    "bootstrap resamples": 1000,
    "confidence level": 0.95,
    "seed": 42,
}, name="production defaults").to_frame())

## 3. Run the exact production command

In [ ]:
if RUN_TRACE:
    exit_code = run_hydra(["command=causal_trace", f"model={MODEL}", *OVERRIDES])
    if exit_code != 0:
        raise RuntimeError(f"causal-trace exited with status {exit_code}")

## 4. Load a completed run

In [ ]:
if RUN_DIR is None:
    candidates = [p.parent for p in OUTPUT_ROOT.glob("*/summary.json")]
    if not candidates:
        raise FileNotFoundError(
            f"No completed run below {OUTPUT_ROOT}. Set RUN_TRACE=True or assign RUN_DIR."
        )
    RUN_DIR = max(candidates, key=lambda p: (p / "summary.json").stat().st_mtime_ns)
RUN_DIR = Path(RUN_DIR).resolve()

summary = json.loads((RUN_DIR / "summary.json").read_text())
selection = json.loads((RUN_DIR / "selection.json").read_text())
discovery = pd.read_csv(RUN_DIR / "discovery_windows.csv")
confirmation = pd.read_csv(RUN_DIR / "confirmation_windows.csv")
aggregate = pd.read_csv(RUN_DIR / "aggregate_windows.csv")
facts = pd.read_json(RUN_DIR / "fact_results.jsonl", lines=True)
rejections_path = RUN_DIR / "rejections.csv"
rejections = pd.read_csv(rejections_path) if rejections_path.stat().st_size else pd.DataFrame()

assert len(facts) == int(summary["num_valid_facts"])
assert summary["selected_trace_center"] == selection["selected_trace_center"]
display(pd.Series({
    "run": str(RUN_DIR),
    "model": summary["model"],
    "scanned": summary["num_dataset_examples_scanned"],
    "accepted": summary["num_valid_facts"],
    "rejected": summary["num_rejected"],
    "discovery facts": summary["num_discovery_facts"],
    "confirmation facts": summary["num_confirmation_facts"],
    "confirmed": summary["confirmation_passed"],
    "selected center": summary["selected_trace_center"],
}, name="run summary").to_frame())

## 5. Discovery and held-out confirmation

In [ ]:
x = aggregate["window_center"].to_numpy(dtype=int)
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x, aggregate["mean_ie"], color="#9ecae1", alpha=0.45, label="all facts")
ax.plot(x, discovery["mean_ie"], "--", color="#555555", label="discovery")
ax.plot(x, confirmation["mean_ie"], color="#08519c", linewidth=2, label="confirmation")
ax.fill_between(
    x,
    confirmation["mean_ie_ci_lower"].to_numpy(float),
    confirmation["mean_ie_ci_upper"].to_numpy(float),
    color="#3182bd",
    alpha=0.18,
    label="confirmation 95% CI",
)
region = summary.get("confirmed_region_centers", [])
if region:
    ax.axvspan(min(region) - 0.5, max(region) + 0.5, color="#31a354", alpha=0.16, label="confirmed region")
selected = summary.get("selected_trace_center")
if selected is not None:
    ax.axvline(selected, color="#006d2c", linewidth=2.5, label=f"selected {selected}")
reference = summary.get("configured_reference_layer")
if reference is not None:
    ax.axvline(reference, color="#756bb1", linestyle=":", linewidth=2, label=f"config reference {reference}")
ax.axhline(0, color="black", linewidth=0.8)
ax.set(xlabel="MLP window center", ylabel="mean paired indirect effect", title="Production causal-trace selection")
ax.legend(ncol=2)
fig.tight_layout()
plt.show()

## 6. Per-fact recovery heatmap

In [ ]:
matrix = np.stack(facts["window_mean_ie"].to_numpy())
fig, ax = plt.subplots(figsize=(12, max(4, min(10, len(facts) * 0.08))))
image = ax.imshow(matrix, aspect="auto", interpolation="nearest", cmap="RdBu_r")
ax.set(xlabel="MLP window center", ylabel="accepted fact", title="Paired indirect effect by fact and window")
ax.set_xticks(np.arange(len(x))[::max(1, len(x) // 12)])
ax.set_xticklabels(x[::max(1, len(x) // 12)])
fig.colorbar(image, ax=ax, label="mean IE across noise draws")
fig.tight_layout()
plt.show()

## 7. Fact eligibility diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(facts["clean_probability"], facts["mean_corrupt_probability"], alpha=0.65)
limit = float(max(facts["clean_probability"].max(), facts["mean_corrupt_probability"].max()))
axes[0].plot([0, limit], [0, limit], color="black", linestyle="--", linewidth=1)
axes[0].set(xlabel="clean target probability", ylabel="corrupt target probability", title="Accepted facts")
if rejections.empty:
    axes[1].text(0.5, 0.5, "No rejected facts saved", ha="center", va="center")
    axes[1].set_axis_off()
else:
    reasons = rejections["reason"].str.split(":").str[0].value_counts().head(10).sort_values()
    reasons.plot.barh(ax=axes[1], color="#de2d26")
    axes[1].set(xlabel="count", ylabel="rejection reason", title="Top rejection reasons")
fig.tight_layout()
plt.show()

## 8. Selected-region numbers

In [ ]:
region_table = confirmation[confirmation["window_center"].isin(summary.get("confirmed_region_centers", []))].copy()
columns = [
    "window_center", "window_layers", "mean_ie", "mean_ie_ci_lower",
    "mean_ie_ci_upper", "median_ie", "normalized_recovery_mean",
]
display(region_table[[c for c in columns if c in region_table.columns]])
display(pd.DataFrame(selection.get("confirmation_regions", [])))